#Parte 3: Optimización y Gobernanza (M3)

##Ejercicio 3.1: Benchmark OPTIMIZE + ZORDER — GUIDED

In [0]:
%sql
-- ANTES: ejecutar y anotar el tiempo
SELECT partido, COUNT(*) as cnt, ROUND(AVG(precio), 2) as avg_precio
FROM bootcamp.semantica.v_propiedades_completa
WHERE moneda = 'USD'
GROUP BY partido
ORDER BY cnt DESC;



In [0]:
%sql
-- OPTIMIZE reorganiza small files + ZORDER coloca datos con zona_id similar en los mismos archivos
OPTIMIZE bootcamp.gold.fact_propiedades ZORDER BY (zona_id);

In [0]:
-- DESPUÉS: misma query. ZORDER permite data skipping sobre zona_id
SELECT partido, COUNT(*) as cnt, ROUND(AVG(precio), 2) as avg_precio
FROM bootcamp.semantica.v_propiedades_completa
WHERE moneda = 'USD'
GROUP BY partido
ORDER BY cnt DESC;

### Ejercicio 3.2 — INDEPENDENT

In [0]:
%sql
-- Ver versiones antes del VACUUM
DESCRIBE HISTORY bootcamp.gold.fact_propiedades;

In [0]:
%sql
-- VACUUM elimina archivos que ya no son referenciados por versiones dentro del período de retención
VACUUM bootcamp.gold.fact_propiedades RETAIN 168 HOURS;

In [0]:
-- Respuestas:
-- 1. Depende de cuántas operaciones se hicieron (ver output de DESCRIBE HISTORY)
-- 2. VACUUM con 0 horas elimina TODOS los archivos de versiones anteriores.
--    Time Travel deja de funcionar para cualquier versión previa.
--    Requiere desactivar el safety check (muy peligroso en producción).
-- 3. Podés seguir usando Time Travel para versiones dentro del período de retención.
--    Las versiones más viejas que 168 horas pierden sus archivos.
DESCRIBE HISTORY bootcamp.gold.fact_propiedades;

In [0]:
%sql
SHOW GRANTS ON TABLE bootcamp.gold.fact_propiedades;

-- Comandos de gobernanza (ejecutar si tenés permisos de admin):
-- 1. GRANT SELECT ON VIEW bootcamp.semantica.v_propiedades_completa TO `analistas`;
-- 2. REVOKE MODIFY ON TABLE bootcamp.gold.fact_propiedades FROM `analistas`;
-- 3. GRANT SELECT ON SCHEMA bootcamp.semantica TO `analistas`;
--    (GRANT USAGE no disponible en privilege version 1.0 del metastore)

-- ¿Por qué dar acceso a la view pero no a la tabla?
-- La view es la interfaz controlada: muestra solo lo que el consumidor necesita.
-- Si le das acceso directo a la tabla, puede ver columnas sensibles,
-- hacer JOINs incorrectos, o modificar datos y romper las optimizaciones.